# Fine-tune the Laya head for Tetris

Trains the decision head of [Laya](https://huggingface.co/convaiinnovations/laya)
on the heuristic-labelled `choice` questions that `bin/make_dataset.dart`
writes, then exports a safetensors head that `DecisionEngine` loads. The
ModernBERT encoder stays frozen: each question is encoded once and only the
head trains. Setup, run times and results are in [README.md](README.md).

## Settings

`DATASET_DIR` holds `train.jsonl` and `val.jsonl`. `TRAIN_ROWS` and
`VAL_ROWS` keep only the first rows of each file for a quick run; `None` uses
every row.

The defaults are the recipe measured in the README: 8 epochs of AdamW
(learning rate 3e-4, weight decay 0.01) with 100 linear warmup steps and a
cosine decay, batches of 32, and soft targets `softmax(h / TAU)` over each
question's heuristic values. `TAU = 0` trains on the dataset's `target`
instead, which splits the probability evenly over the heuristic-best options.

`OUT_DTYPE` is `"F32"` or `"F16"`.

In [ ]:
from pathlib import Path

DATASET_DIR = Path("../dataset")
TRAIN_ROWS = None
VAL_ROWS = None

EPOCHS = 8
LR = 3e-4
TAU = 0.3
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 100
BATCH_SIZE = 32
SEED = 0

OUT_PATH = Path("laya-head-tetris.safetensors")
OUT_DTYPE = "F32"

CHECKPOINT = "convaiinnovations/laya"
REVISION = "1c5edc17a7acd8701df6fc341c0d179f1c62c982"

## Load Laya

Downloads the checkpoint at `REVISION`, the revision llamadart's Laya parity
fixture was made from, and moves the model in FP32 to MPS when available,
else CUDA, else the CPU.

In [ ]:
import json
import math
import random
import time

import torch
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from safetensors.torch import save_file

import laya
from laya.common import QTYPES, build_sequence

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

checkpoint_dir = Path(snapshot_download(
    CHECKPOINT,
    revision=REVISION,
    allow_patterns=["rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*"],
))
config_text = (checkpoint_dir / "rl_agent_config.json").read_text()

torch.manual_seed(SEED)
random.seed(SEED)
agent = laya.load(str(checkpoint_dir), device="cpu")
model = agent.model.float().to(device)
tok = agent.tok
cfg = agent.cfg
hidden_size = model.type_emb.embedding_dim
print(f"laya {laya.__version__}, torch {torch.__version__}, device {device}")

## Build sequences

`build_sequence` turns each state and question into Laya's input,
`[CLS] question [SEP] [MASK] option ... [SEP] state [SEP]`, and returns the
positions of the option markers.

In [ ]:
def load_rows(path, limit):
    items = []
    with open(path) as f:
        for line in f:
            if limit is not None and len(items) == limit:
                break
            row = json.loads(line)
            ids, markers = build_sequence(
                tok,
                row["state"],
                agent._to_internal(row["q"]),
                max_len=cfg["max_len"],
                head_max_len=cfg["head_max_len"],
            )
            assert len(markers) == len(row["target"]), row
            items.append({"ids": ids, "markers": markers, "target": row["target"], "h": row["h"]})
    return items


train = load_rows(DATASET_DIR / "train.jsonl", TRAIN_ROWS)
val = load_rows(DATASET_DIR / "val.jsonl", VAL_ROWS)
lengths = [len(it["ids"]) for it in train + val]
print(f"train {len(train)}, val {len(val)}; tokens per sequence: "
      f"mean {sum(lengths) / len(lengths):.0f}, max {max(lengths)}")

## Encode once

The encoder is frozen, so its last hidden states are computed once, in
batches of 32 sequences of similar length, and kept in memory in FP16.

In [ ]:
@torch.no_grad()
def encode(items, batch_size=32):
    model.encoder.eval()
    order = sorted(range(len(items)), key=lambda i: len(items[i]["ids"]))
    start = time.time()
    for s in range(0, len(order), batch_size):
        idx = order[s : s + batch_size]
        length = max(len(items[i]["ids"]) for i in idx)
        ids = torch.full((len(idx), length), tok.pad_token_id, dtype=torch.long)
        mask = torch.zeros((len(idx), length), dtype=torch.long)
        for j, i in enumerate(idx):
            n = len(items[i]["ids"])
            ids[j, :n] = torch.tensor(items[i]["ids"])
            mask[j, :n] = 1
        hidden = model.encoder(input_ids=ids.to(device), attention_mask=mask.to(device)).last_hidden_state
        hidden = hidden.to("cpu", torch.float16)
        for j, i in enumerate(idx):
            items[i]["hidden"] = hidden[j, : len(items[i]["ids"])].clone()
    print(f"encoded {len(items)} sequences in {time.time() - start:.0f} s")


encode(val)
encode(train)

## Head and metrics

`head_forward` is the part of Laya's `DecisionModel.forward` after the encoder,
for `choice` questions: the type embedding, the head's transformer layers, and
the scorer at each option marker.

A validation question counts as right when the head's top option is one of the
heuristic-best options. Regret is the best heuristic value minus the value of
the option the head picked. Chance is the accuracy of a uniform random pick.

In [ ]:
def collate(items):
    length = max(len(it["ids"]) for it in items)
    k = max(len(it["markers"]) for it in items)
    hidden = torch.zeros((len(items), length, hidden_size))
    mask = torch.zeros((len(items), length), dtype=torch.bool)
    markers = torch.zeros((len(items), k), dtype=torch.long)
    marker_mask = torch.zeros((len(items), k), dtype=torch.bool)
    target = torch.zeros((len(items), k))
    for j, it in enumerate(items):
        n, m = len(it["ids"]), len(it["markers"])
        hidden[j, :n] = it["hidden"].float()
        mask[j, :n] = True
        markers[j, :m] = torch.tensor(it["markers"])
        marker_mask[j, :m] = True
        target[j, :m] = torch.softmax(torch.tensor(it["h"]) / TAU, -1) if TAU > 0 else torch.tensor(it["target"])
    return [t.to(device) for t in (hidden, mask, markers, marker_mask, target)]


def head_forward(hidden, mask, markers, marker_mask):
    qtype = torch.full((hidden.size(0),), QTYPES["choice"], device=device)
    h = hidden + model.type_emb(qtype)[:, None, :]
    for layer in model.head.layers:
        h = layer(h, src_key_padding_mask=~mask)
    rows = torch.gather(h, 1, markers[:, :, None].expand(-1, -1, h.size(-1)))
    return model.scorer(rows).squeeze(-1).float().masked_fill(~marker_mask, -1e4)


@torch.no_grad()
def evaluate(items, label):
    model.head.eval()
    model.scorer.eval()
    right = regret = 0.0
    for s in range(0, len(items), 64):
        chunk = items[s : s + 64]
        hidden, mask, markers, marker_mask, _ = collate(chunk)
        picks = head_forward(hidden, mask, markers, marker_mask).argmax(-1).tolist()
        for it, p in zip(chunk, picks):
            right += it["target"][p] > 0
            regret += max(it["h"]) - it["h"][p]
    chance = sum(sum(t > 0 for t in it["target"]) / len(it["target"]) for it in items) / len(items)
    result = {"accuracy": right / len(items), "regret": regret / len(items)}
    print(f"{label}: accuracy {result['accuracy']:.3f} (chance {chance:.3f}), "
          f"mean regret {result['regret']:.3f}")
    return result


base = evaluate(val, "base head")

## Train

Only `head.*`, `type_emb.*` and `scorer.*` train; the act head keeps the
checkpoint's weights. Gradients are clipped to norm 1. After each epoch the
head is scored on the validation set, and the epoch with the best accuracy is
kept. Running this cell again continues from the current weights; run the
cells from **Load Laya** on to start over.

In [ ]:
params = [p for n, p in model.named_parameters() if n.startswith(("head.", "type_emb.", "scorer."))]
for p in model.parameters():
    p.requires_grad_(False)
for p in params:
    p.requires_grad_(True)
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
batches = math.ceil(len(train) / BATCH_SIZE)
steps = EPOCHS * batches
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda s: min(1.0, (s + 1) / WARMUP_STEPS) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps))),
)

best, best_state = None, None
for epoch in range(1, EPOCHS + 1):
    random.shuffle(train)
    model.head.train()
    model.scorer.train()
    start, total = time.time(), 0.0
    for s in range(0, len(train), BATCH_SIZE):
        hidden, mask, markers, marker_mask, target = collate(train[s : s + BATCH_SIZE])
        logits = head_forward(hidden, mask, markers, marker_mask)
        loss = -(target * F.log_softmax(logits, -1)).sum(-1).mean()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optimizer.step()
        scheduler.step()
        total += loss.item()
    print(f"epoch {epoch}: mean loss {total / batches:.3f} in {time.time() - start:.0f} s")
    result = evaluate(val, f"epoch {epoch}")
    if best is None or result["accuracy"] > best["accuracy"]:
        best = {**result, "epoch": epoch}
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                      if not k.startswith("encoder.")}

print(f"base head:  accuracy {base['accuracy']:.3f}, mean regret {base['regret']:.3f}")
print(f"tuned head: accuracy {best['accuracy']:.3f}, mean regret {best['regret']:.3f} (epoch {best['epoch']})")

## Export

Writes the kept epoch's head: every tensor except `encoder.*`, under Laya's
PyTorch names, with the checkpoint's `rl_agent_config.json` as `laya.config`
metadata, so `DecisionEngine.load` needs no `configPath`.

In [ ]:
dtype = {"F32": torch.float32, "F16": torch.float16}[OUT_DTYPE]
tensors = {k: v.to(dtype).contiguous() for k, v in best_state.items()}
save_file(tensors, OUT_PATH, metadata={"laya.config": config_text})
print(f"wrote {OUT_PATH.resolve()}: {len(tensors)} {OUT_DTYPE} tensors, {OUT_PATH.stat().st_size / 1e6:.0f} MB")